# Persian Book Cover Lens

## Persian Book Title Recognition from Cover Images using Vision-Language Models

This notebook implements a course project for extracting Persian book titles from book cover images.

The main GPU experiment uses **Qwen2.5-VL** as a Vision-Language Model. Because full VLM fine-tuning is not practical on a CPU-only Colab runtime, the notebook also includes a CPU-safe `dev_demo` mode. The development mode validates the full project pipeline without pretending to produce real model performance.

**Repository:** `vlm_persian_book_cover_lens`  
**Notebook:** `PersianBookCoverLens.ipynb`

## 1. Installation

Run this cell once in Google Colab. After installation, restart the runtime if Colab asks for it.

Recommended workflow:

1. Run the installation cell.
2. Restart runtime once.
3. Continue from Section 2.
4. Do not repeatedly reinstall packages unless the runtime has been reset.

In [ ]:
# Colab installation cell
# Run once, then restart the runtime if needed.

INSTALL_PACKAGES = False  # Set to True in Colab when dependencies are missing.

if INSTALL_PACKAGES:
    !pip -q uninstall -y pillow PIL || true
    !pip -q install --no-cache-dir --force-reinstall "Pillow==11.3.0"
    !pip -q install --upgrade --upgrade-strategy only-if-needed         "pandas==2.2.2"         "datasets>=3.0.0"         "transformers>=4.53.0"         "accelerate>=1.8.0"         "peft>=0.13.0"         "bitsandbytes>=0.43.0"         "qwen-vl-utils>=0.0.8"         "rapidfuzz>=3.9.0"         "matplotlib>=3.7.0"         "tqdm>=4.66.0"
else:
    print("Package installation skipped. Set INSTALL_PACKAGES=True if running in a fresh Colab runtime.")

## 2. Imports, Reproducibility, and Global Configuration

There are two execution modes:

- `dev_demo`: safe for CPU-only Colab; validates data, metrics, outputs, and plots.
- `qwen_gpu`: real Qwen2.5-VL baseline inference and LoRA/QLoRA fine-tuning; requires CUDA GPU.

For your current CPU-only Colab test, keep:

```python
RUN_MODE = "dev_demo"
```

For the real final VLM run on a GPU runtime, use:

```python
RUN_MODE = "qwen_gpu"
FAST_DEV_RUN = False
```

In [ ]:
import os
import re
import random
import unicodedata
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from rapidfuzz.distance import Levenshtein

try:
    import torch
except Exception as exc:
    torch = None
    print("PyTorch import failed:", exc)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

DATASET_ID = "shenasa/bookroom-persian-book-covers-and-titles"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# Main switch:
# - "dev_demo": CPU-safe project demonstration, no heavy VLM execution.
# - "qwen_gpu": real VLM baseline + optional LoRA/QLoRA fine-tuning, requires CUDA.
RUN_MODE = "dev_demo"

# Quick smoke test option. For a final GPU run, set this to False.
FAST_DEV_RUN = True

# Real GPU controls. These are used only when RUN_MODE == "qwen_gpu".
RUN_BASELINE_INFERENCE = True
RUN_FINETUNING = True
RUN_FINETUNED_INFERENCE = True

if FAST_DEV_RUN:
    TRAIN_LIMIT = 16
    EVAL_LIMIT = 8
    NUM_TRAIN_EPOCHS = 1
else:
    TRAIN_LIMIT = 1000
    EVAL_LIMIT = 200
    NUM_TRAIN_EPOCHS = 1

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPT = (
    "You are a Persian OCR assistant. Extract only the main Persian book title "
    "visible on the cover image. Return only the title text, without explanation."
)

print("RUN_MODE:", RUN_MODE)
print("FAST_DEV_RUN:", FAST_DEV_RUN)
print("TRAIN_LIMIT:", TRAIN_LIMIT)
print("EVAL_LIMIT:", EVAL_LIMIT)
print("CUDA available:", bool(torch is not None and torch.cuda.is_available()))

## 3. Load Dataset

The dataset contains book cover images and Persian title labels. The notebook selects a training subset and a test subset according to the assignment setup.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_ID)
print(dataset)

train_ds = dataset["train"].select(range(min(TRAIN_LIMIT, len(dataset["train"]))))
eval_ds = dataset["test"].select(range(min(EVAL_LIMIT, len(dataset["test"]))))

print("Train subset:", len(train_ds))
print("Eval subset:", len(eval_ds))
print("Columns:", train_ds.column_names)

## 4. Inspect a Few Samples

This cell displays a few covers and their ground-truth Persian titles.

In [ ]:
def show_samples(ds, n: int = 3):
    n = min(n, len(ds))
    for i in range(n):
        sample = ds[i]
        print(f"Index: {i}")
        print("Title:", sample["text"])
        display(sample["image"].resize((220, 320)))

show_samples(eval_ds, n=3)

## 5. Persian Text Normalization and Evaluation Metrics

This project computes the metrics required by the assignment:

- Raw Exact Match
- Normalized Exact Match
- Levenshtein Similarity
- Word-level F1

Normalization is important for Persian text because OCR outputs may use Arabic variants of Persian characters, inconsistent spacing, diacritics, or punctuation.

In [ ]:
PERSIAN_ARABIC_MAP = str.maketrans({
    "ي": "ی",
    "ى": "ی",
    "ك": "ک",
    "ؤ": "و",
    "إ": "ا",
    "أ": "ا",
    "ٱ": "ا",
    "ة": "ه",
    "ۀ": "ه",
})

DIACRITICS_RE = re.compile(r"[ً-ٰٟۖ-ۭ]")
PUNCT_RE = re.compile(r"[\.,;:!؟?،؛\-_/\|\(\)\[\]{}<>"'«»“”‘’]+")
SPACE_RE = re.compile(r"\s+")


def normalize_persian_text(text: Any) -> str:
    """Normalize Persian text for fairer OCR evaluation."""
    if text is None:
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(PERSIAN_ARABIC_MAP)
    text = DIACRITICS_RE.sub("", text)
    text = PUNCT_RE.sub(" ", text)
    text = SPACE_RE.sub(" ", text).strip()
    return text


def word_level_f1(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_persian_text(prediction).split()
    gt_tokens = normalize_persian_text(ground_truth).split()
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    pred_counts = {}
    gt_counts = {}
    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1
    for token in gt_tokens:
        gt_counts[token] = gt_counts.get(token, 0) + 1
    overlap = sum(min(pred_counts.get(tok, 0), gt_counts.get(tok, 0)) for tok in gt_counts)
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gt_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    rows = []
    for pred, ref in zip(predictions, references):
        pred_norm = normalize_persian_text(pred)
        ref_norm = normalize_persian_text(ref)
        raw_em = int(str(pred).strip() == str(ref).strip())
        norm_em = int(pred_norm == ref_norm)
        if pred_norm == "" and ref_norm == "":
            lev_sim = 1.0
        elif pred_norm == "" or ref_norm == "":
            lev_sim = 0.0
        else:
            lev_sim = float(Levenshtein.normalized_similarity(pred_norm, ref_norm))
        f1 = word_level_f1(pred, ref)
        rows.append({
            "exact_match_raw": raw_em,
            "exact_match_normalized": norm_em,
            "levenshtein_similarity": lev_sim,
            "word_level_f1": f1,
        })
    df = pd.DataFrame(rows)
    return {col: float(df[col].mean()) for col in df.columns}


def build_prediction_df(stage: str, predictions: List[str], references: List[str]) -> pd.DataFrame:
    df = pd.DataFrame({
        "stage": stage,
        "prediction": predictions,
        "ground_truth": references,
    })
    df["prediction_normalized"] = df["prediction"].apply(normalize_persian_text)
    df["ground_truth_normalized"] = df["ground_truth"].apply(normalize_persian_text)
    return df

# Tiny sanity check for metric functions
_metric_test = compute_metrics(["کتاب فارسی", "عنوان اشتباه"], ["کتاب فارسی", "عنوان درست"])
_metric_test

## 6. Runtime Guard

This section prevents accidental CPU execution of the heavy Qwen2.5-VL path. If you select `qwen_gpu` without a CUDA GPU, the notebook stops clearly instead of producing weak or misleading CPU predictions.

In [ ]:
def cuda_is_available() -> bool:
    return bool(torch is not None and torch.cuda.is_available())

if RUN_MODE not in {"dev_demo", "qwen_gpu"}:
    raise ValueError("RUN_MODE must be either 'dev_demo' or 'qwen_gpu'.")

if RUN_MODE == "qwen_gpu" and not cuda_is_available():
    raise RuntimeError(
        "RUN_MODE='qwen_gpu' requires a CUDA GPU runtime. "
        "Switch Colab runtime to GPU, or use RUN_MODE='dev_demo' for CPU-safe pipeline validation."
    )

print("Runtime guard passed.")

## 7. Development Demo Mode

This mode is intentionally CPU-safe. It does **not** run the Qwen model and does **not** claim real model accuracy.

Its purpose is to validate:

- dataset loading
- output schemas
- metric functions
- CSV saving
- comparison plot generation
- notebook flow for presentation

Real performance numbers should be generated using `RUN_MODE = "qwen_gpu"`.

In [ ]:
def run_dev_demo(eval_dataset) -> Dict[str, pd.DataFrame]:
    references = [sample["text"] for sample in eval_dataset]

    # These are explicit placeholders, not model predictions.
    baseline_predictions = ["[DEV DEMO] Base Qwen2.5-VL was not executed on CPU." for _ in references]
    finetuned_predictions = ["[DEV DEMO] Fine-tuned Qwen2.5-VL was not executed on CPU." for _ in references]

    baseline_df = build_prediction_df("before_finetuning_dev_demo", baseline_predictions, references)
    finetuned_df = build_prediction_df("after_finetuning_dev_demo", finetuned_predictions, references)

    return {
        "baseline_df": baseline_df,
        "finetuned_df": finetuned_df,
    }

if RUN_MODE == "dev_demo":
    demo_outputs = run_dev_demo(eval_ds)
    baseline_df = demo_outputs["baseline_df"]
    finetuned_df = demo_outputs["finetuned_df"]
    display(baseline_df.head(5))
    display(finetuned_df.head(5))
else:
    baseline_df = None
    finetuned_df = None
    print("Skipping dev_demo section because RUN_MODE is qwen_gpu.")

## 8. Qwen2.5-VL Baseline Inference — GPU Track

This section runs the base Qwen2.5-VL model before fine-tuning. It only runs when:

```python
RUN_MODE = "qwen_gpu"
RUN_BASELINE_INFERENCE = True
```

In [ ]:
def load_qwen_model_and_processor(load_in_4bit: bool = True):
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

    if load_in_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
    else:
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )

    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    return model, processor


def predict_title_qwen(model, processor, image, prompt: str = PROMPT, max_new_tokens: int = 48) -> str:
    from qwen_vl_utils import process_vision_info

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return output_text.strip()


def run_qwen_inference(model, processor, eval_dataset, stage: str) -> pd.DataFrame:
    predictions = []
    references = []
    for sample in tqdm(eval_dataset, desc=f"Running Qwen inference: {stage}"):
        pred = predict_title_qwen(model, processor, sample["image"])
        predictions.append(pred)
        references.append(sample["text"])
    return build_prediction_df(stage, predictions, references)

if RUN_MODE == "qwen_gpu" and RUN_BASELINE_INFERENCE:
    model, processor = load_qwen_model_and_processor(load_in_4bit=True)
    baseline_df = run_qwen_inference(model, processor, eval_ds, stage="before_finetuning")
    baseline_df.to_csv(OUTPUT_DIR / "baseline_predictions.csv", index=False)
    display(baseline_df.head(10))
elif RUN_MODE == "qwen_gpu":
    print("Baseline inference skipped by configuration.")
else:
    print("Qwen baseline inference skipped in dev_demo mode.")

## 9. LoRA/QLoRA Fine-tuning — GPU Track

This section fine-tunes Qwen2.5-VL using PEFT LoRA adapters. It only runs when:

```python
RUN_MODE = "qwen_gpu"
RUN_FINETUNING = True
```

The training setup is intentionally conservative for Colab GPU:

- 4-bit loading
- batch size 1
- gradient accumulation
- LoRA adapters
- no full model fine-tuning

In [ ]:
def prepare_training_example(processor, sample: Dict[str, Any]) -> Dict[str, Any]:
    from qwen_vl_utils import process_vision_info

    title = str(sample["text"])
    image = sample["image"]
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": title},
            ],
        },
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    image_inputs, video_inputs = process_vision_info(messages)
    encoded = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    encoded = {k: v.squeeze(0) for k, v in encoded.items()}
    labels = encoded["input_ids"].clone()

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100

    # Mask common visual placeholder tokens if available.
    for token_attr in ["image_token", "video_token"]:
        token = getattr(processor, token_attr, None)
        if token is not None:
            token_id = processor.tokenizer.convert_tokens_to_ids(token)
            if token_id is not None and token_id >= 0:
                labels[labels == token_id] = -100

    encoded["labels"] = labels
    return encoded


class QwenBookTitleDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return prepare_training_example(self.processor, self.dataset[idx])


def qwen_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    from torch.nn.utils.rnn import pad_sequence

    output = {}
    for key in batch[0].keys():
        values = [item[key] for item in batch]
        if key in {"input_ids", "attention_mask", "labels"}:
            pad_value = -100 if key == "labels" else 0
            output[key] = pad_sequence(values, batch_first=True, padding_value=pad_value)
        else:
            # pixel_values and image_grid_thw normally have compatible shapes when batch_size=1.
            # This project uses batch size 1 for stable Colab training.
            output[key] = torch.stack(values, dim=0) if values[0].ndim > 0 else torch.tensor(values)
    return output


def setup_lora_model(model):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


if RUN_MODE == "qwen_gpu" and RUN_FINETUNING:
    from transformers import Trainer, TrainingArguments

    # If baseline section was skipped, load model here.
    if "model" not in globals() or model is None:
        model, processor = load_qwen_model_and_processor(load_in_4bit=True)

    model = setup_lora_model(model)
    train_dataset_for_trainer = QwenBookTitleDataset(train_ds, processor)

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "qwen_lora_training"),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=5,
        save_strategy="epoch",
        save_total_limit=1,
        bf16=torch.cuda.is_available(),
        fp16=False,
        optim="paged_adamw_8bit",
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset_for_trainer,
        data_collator=qwen_collate_fn,
    )

    trainer.train()
    adapter_dir = OUTPUT_DIR / "qwen_lora_adapter"
    model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)
    print("Saved LoRA adapter to:", adapter_dir)
elif RUN_MODE == "qwen_gpu":
    print("Fine-tuning skipped by configuration.")
else:
    print("Qwen fine-tuning skipped in dev_demo mode.")

## 10. Evaluation After Fine-tuning — GPU Track

This section runs inference again using the fine-tuned LoRA adapter and compares the results with the baseline.

In [ ]:
if RUN_MODE == "qwen_gpu" and RUN_FINETUNED_INFERENCE:
    if "model" not in globals() or model is None:
        raise RuntimeError("Model is not loaded. Run the baseline/fine-tuning sections first.")

    finetuned_df = run_qwen_inference(model, processor, eval_ds, stage="after_finetuning")
    finetuned_df.to_csv(OUTPUT_DIR / "finetuned_predictions.csv", index=False)
    display(finetuned_df.head(10))
elif RUN_MODE == "qwen_gpu":
    print("Fine-tuned inference skipped by configuration.")
else:
    print("Fine-tuned inference skipped in dev_demo mode.")

## 11. Metrics Summary and Comparison Plot

This section computes and saves the final metrics table and comparison chart.

In `dev_demo` mode, the metrics are computed on explicit placeholders, so they should be interpreted only as pipeline-validation outputs.

In `qwen_gpu` mode, the metrics represent real model predictions before and after fine-tuning.

In [ ]:
def summarize_stage(df: pd.DataFrame, stage_name: str) -> Dict[str, Any]:
    metrics = compute_metrics(df["prediction"].tolist(), df["ground_truth"].tolist())
    return {
        "stage": stage_name,
        "n": len(df),
        **metrics,
    }

if baseline_df is None or finetuned_df is None:
    raise RuntimeError("Prediction dataframes are missing. Run either dev_demo or qwen_gpu prediction sections first.")

metrics_summary = pd.DataFrame([
    summarize_stage(baseline_df, baseline_df["stage"].iloc[0]),
    summarize_stage(finetuned_df, finetuned_df["stage"].iloc[0]),
])

metrics_summary.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
baseline_df.to_csv(OUTPUT_DIR / "baseline_predictions.csv", index=False)
finetuned_df.to_csv(OUTPUT_DIR / "finetuned_predictions.csv", index=False)
finetuned_df.head(20).to_csv(OUTPUT_DIR / "sample_predictions_after_finetuning.csv", index=False)

display(metrics_summary)
print("Sample predictions after fine-tuning:")
display(finetuned_df.head(10))

In [ ]:
metric_cols = [
    "exact_match_raw",
    "exact_match_normalized",
    "levenshtein_similarity",
    "word_level_f1",
]

plot_df = metrics_summary.set_index("stage")[metric_cols].T
ax = plot_df.plot(kind="bar", figsize=(11, 5))
ax.set_title("Before vs After Fine-tuning: Metric Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "metrics_comparison.png", dpi=160)
plt.show()

print("Saved outputs to:", OUTPUT_DIR.resolve())

## 12. Final Notes for Reporting

If the notebook was run in `dev_demo` mode, report it as a CPU-safe development demonstration:

> Due to CPU-only runtime limitations, the notebook was executed in development demonstration mode. The full GPU-based Qwen2.5-VL baseline inference and LoRA/QLoRA fine-tuning implementation is included, but actual VLM training requires a CUDA GPU runtime.

If the notebook was run in `qwen_gpu` mode, report the real metrics from `outputs/metrics_summary.csv` and include the plot from `outputs/metrics_comparison.png`.